[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rudrite/kernels/blob/main/labs/pytorch/lab-p3-guards-breaks-counters.ipynb)

# LAB·P3 · Guards, breaks, counters

**Hardware:** any machine, but you need Python 3.11 or newer and a torch build that supports `torch.compile`. The next cell checks both before anything else runs.

`torch.compile` reads your function's bytecode, captures the tensor ops into a graph, and installs guards on the things that could invalidate that graph: shapes, dtypes, types, closed-over values. A call whose guards still hold reuses the compiled graph; a guard miss captures again. This lab counts that behavior directly through dynamo's own counters, watches a data-dependent branch turn into a graph break instead of an error, and closes by exporting a module to the ATen graph both bridges in chapter 10 want.

Before every reveal cell there is an empty "your prediction" cell above it. Write your answer there, then run the reveal and compare.

In [ ]:
import sys
import torch

print(sys.version)
print(torch.__version__)
assert sys.version_info >= (3, 11), "this lab needs Python 3.11 or newer for the dynamo/export cells"

## The counters

`torch._dynamo.utils.counters` is dynamo's own bookkeeping: `calls_captured` counts every call that went through the compiled path, and `unique_graphs` counts how many distinct graphs dynamo actually had to build. Two calls with the same guards should reuse one graph; a call with a new shape should not.

**your prediction:**

`f` is compiled with `backend="eager"`. Predict `counters["stats"]` after calling `f` twice with tensors of shape `(4,)`. Then predict how it changes after one more call with shape `(5,)`.

In [ ]:
from torch._dynamo.utils import counters

@torch.compile(backend="eager")
def f(x):
    return torch.sin(x) + 1

f(torch.randn(4)); f(torch.randn(4))
print(dict(counters["stats"]))   # {'calls_captured': 2, 'unique_graphs': 1}
f(torch.randn(5))
print(dict(counters["stats"]))   # {'calls_captured': 4, 'unique_graphs': 2}

Two same-shape calls share one graph: `unique_graphs` stays at 1 while `calls_captured` counts both of them. The shape-5 call trips a guard that named shape 4, so dynamo captures a second graph, and both counters jump. Contrast this with the JAX path's trace-once model: there is no single trace here to reuse or discard, only a cache keyed on whichever guards happen to hold.

## Python side effects replay

Capture only touches the tensor math. Anything else your function does as ordinary Python, including a side effect like mutating a global, still runs on every call, compiled or not.

**your prediction:** `h` increments a global counter and is wrapped in `torch.compile`. Predict what the counter reads after two calls.

In [ ]:
calls = 0

@torch.compile(backend="eager")
def h(x):
    global calls
    calls += 1
    return x + 1

h(torch.randn(3))
h(torch.randn(3))
print(calls)   # 2: dynamo compiled the tensor math and replayed the surrounding Python

The counter reads 2, not 1: dynamo compiled `x + 1` and left the increment as ordinary Python to run again on every call. Nothing about compilation here memoized the side effect away.

## Data-dependent control flow: a break, not an error

The JAX path's `TracerBoolConversionError` has no sibling here. When dynamo hits a branch that depends on a tensor's runtime value, it does not raise: it graph breaks, runs that branch in plain Python, and stitches compiled pieces around it. The cost is silent, fewer and smaller graphs, and `torch._dynamo.explain` is how you count it.

**your prediction:** `g` branches on `x.sum() > 0`. Predict `graph_break_count` from `dynamo.explain(g)(torch.ones(3))`.

In [ ]:
import torch._dynamo as dynamo

def g(x):
    if x.sum() > 0:        # data-dependent: a graph break, not an error
        return x + 1
    return x - 1

print(dynamo.explain(g)(torch.ones(3)).graph_break_count)   # 1

One break: dynamo cannot know which branch a future call will take without running the tensor op that decides it, so it drops out of capture at the `if`, runs the comparison and the chosen branch in Python, and would resume compiling around it in a larger function. No exception, no crash, just a smaller graph than the one you might have expected.

## Below dynamo, ATen

`torch.export` produces a standalone `ExportedProgram`: no Python control flow left, every node a call into an ATen op, the artifact both bridges in chapter 10 consume.

**your prediction:** for a module whose forward is `torch.sin(x).sum()`, name the two ATen ops you expect the exported graph to contain, in order.

In [ ]:
class Sin(torch.nn.Module):
    def forward(self, x):
        return torch.sin(x).sum()

ep = torch.export.export(Sin(), (torch.randn(4),))
print(ep.graph)
# graph():
#     %l_x_ : [num_users=1] = placeholder[target=l_x_]
#     %sin : [num_users=1] = call_function[target=torch.ops.aten.sin.default](args = (%l_x_,), kwargs = {})
#     %sum_1 : [num_users=1] = call_function[target=torch.ops.aten.sum.default](args = (%sin,), kwargs = {})
#     return (sum_1,)

Read it the way the JAX path reads a jaxpr: one placeholder in, one ATen call per line, a return at the bottom. `aten.sin.default` and `aten.sum.default` are the same two ops the module's forward named, now with no Python left around them, stricter than anything dynamo alone produces. This is the whole-graph artifact a serving stack or a cross-compiler handoff wants.

## Exercise

Take three functions of your own and run them through `dynamo.explain`, writing down `graph_break_count` and the guard count for each before you look. Then export one module you compiled earlier and find a case dynamo splits into two graphs but `torch.export` refuses outright; write a sentence on why the two have different tolerances for the same Python.

## Mark it run

Read the two chapters this lab drills: [kernels.rudrite.com/pytorch/dynamo](https://kernels.rudrite.com/pytorch/dynamo) and [kernels.rudrite.com/pytorch/graphs](https://kernels.rudrite.com/pytorch/graphs).